# Portfolio model summary

This notebook explains the final merchant portfolio model and checks the delivered dashboard outputs. It reads the fraud-adjusted final exports and matching input snapshot, plus curated attributes for descriptive comparisons. It does not rebuild or change the model, its selections or its five segment definitions.

The model selects exactly **100 merchants**. Saved results cover one global pool and five category groups at five minimum expected-revenue retention levels: 100%, 99.5%, 99%, 98%, and 95%. Section 6 displays the **complete global list of 100** and **the first 10 merchants in each segment's existing ranked list**, making those requested outputs directly reviewable. The additional portfolio analyses explain the trade-offs and the features distinguishing selected merchants.

Read [02_analysis.ipynb](02_analysis.ipynb) for original missingness, outlier treatment and fraud-score limitations. The implementation is in [fraud_adjusted_portfolios.py](../scripts/fraud_adjusted_portfolios.py) and [11_category_ranking.py](../scripts/11_category_ranking.py); the frozen results and field definitions are documented in the [adjusted handoff](../../README.md#fraud-adjustment-and-findings).


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

# Works from the repository root or any nested notebook directory.
PROJECT_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                     if (p / "code/notebooks").is_dir() and (p / "handoff").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from inside the project repository.")
FINAL_DIR = PROJECT_ROOT / "handoff" / "fraud_adjusted" / "final"

DASHBOARD_FILES = {
    "Overall": "portfolio_dashboard.csv",
    "Arts, Crafts, Gifts & Entertainment": "arts_portfolio_dashboard.csv",
    "Automotive, Outdoor & Recreational Equipment": "auto_portfolio_dashboard.csv",
    "Fashion, Jewellery & Personal Care": "fashion_portfolio_dashboard.csv",
    "Home, Garden & General Supplies": "home_portfolio_dashboard.csv",
    "Technology, Digital & Communications": "tech_portfolio_dashboard.csv",
}

dashboards = {
    label: pd.read_csv(FINAL_DIR / filename, dtype={"merchant_abn": "string"})
    for label, filename in DASHBOARD_FILES.items()
}

display(pd.DataFrame(
    {
        "dashboard": label,
        "rows": len(frame),
        "portfolios": frame["portfolio_id"].nunique(),
        "merchants_per_portfolio": int(frame.groupby("portfolio_id").size().min()),
    }
    for label, frame in dashboards.items()
))
for scope, frame in dashboards.items():
    if not frame["fee_basis"].eq("consumer_fraud_adjusted").all():
        raise ValueError(f"Unexpected return basis in {scope}")


,dashboard,rows,portfolios,merchants_per_portfolio
0,Overall,500,5,100
1,"Arts, Crafts, Gifts & Entertainment",500,5,100
2,"Automotive, Outdoor & Recreational Equipment",500,5,100
3,"Fashion, Jewellery & Personal Care",500,5,100
4,"Home, Garden & General Supplies",500,5,100
5,"Technology, Digital & Communications",500,5,100


## 1. Expected monthly return

The dashboard uses **expected monthly return**, adjusted for both merchant take rate and consumer fraud probability. For merchant $i$ over $M$ complete months:

$$E_i = \frac{1}{M}\sum_{t \in \text{observation window}} v_{it}\frac{\text{take rate}_i}{100}\left(1-\frac{p_{it}}{100}\right)$$

Here $v_{it}$ is transaction value and $p_{it}$ is its consumer fraud score in percentage points. The observation window remains March 2021–September 2022 (19 complete months), including genuine zero months. This is a retrospective return proxy, not net profit or verified losses; merchant fraud is not multiplied in separately.

Portfolio expected return is $E_P=\sum_i z_i E_i$. The baseline selects the 100 eligible merchants with the highest adjusted expected return. Return retention (α) imposes $E_P\geq\alpha E_{baseline}$. `retention` is the target and `portfolio_expected_fee_retention` is achieved retention. Rankings order the selected merchants by adjusted monthly return.

Use `merchant_expected_monthly_adjusted_fees` and `portfolio_expected_monthly_adjusted_fees`. The new files retain legacy `*_gross_fees` aliases for compatibility, but those aliases now also hold adjusted values. Every export must declare `fee_basis=consumer_fraud_adjusted`.

Missing scores were reconstructed with the existing consumer OLS model. Most transactions are outside observed labelled predictor support. These estimates are not a verified point-in-time backtest, and lower modelled risk does not establish future loss reductions.


## 2. Stability (cash-flow variability)

Stability is measured from **26 complete Monday–Sunday weeks** of merchant adjusted-return history. Let $x_t$ be the vector of merchant fees in week $t$. The model estimates a Ledoit–Wolf shrinkage covariance matrix:

$$\widehat{\Sigma}_{shrunk} = (1-\alpha)\widehat{\Sigma}_{sample} + \alpha\,\mathrm{diag}(\widehat{\Sigma}_{sample})$$

Shrinkage reduces unstable off-diagonal covariance estimates while retaining each merchant's variance. For selection vector $z$:

$$SD_P = \sqrt{z^T\widehat{\Sigma}_{shrunk}z}$$

$$CV_P = \frac{SD_P}{\sum_i z_i\bar{x}_i}$$

Here $\bar{x}_i$ is merchant $i$'s historical weekly mean. The coefficient of variation (CV) makes volatility comparable relative to portfolio size: a **lower CV means more stable weekly fee generation**. CV is a ratio and is not constrained to the interval $[0,1]$.

## 3. Customer concentration

Customer concentration measures whether expected portfolio fees depend heavily on a small number of customers. Historical customer–merchant adjusted return are rescaled merchant by merchant so that each merchant's customer allocations sum to its expected monthly fees. This gives exposure $A_{ci}$ from customer $c$ to merchant $i$. Shared customers are combined across all selected merchants:

$$F_c(z) = \sum_i A_{ci}z_i$$

Customer shares and the Herfindahl–Hirschman Index (HHI) are then:

$$s_c(z) = \frac{F_c(z)}{\sum_c F_c(z)}, \qquad HHI_P = \sum_c s_c(z)^2$$

HHI lies between 0 and 1. **Lower HHI means customer exposure is more diversified**; higher HHI means more reliance on a smaller customer base. The reciprocal, $1/HHI$, can be interpreted as the effective number of equally weighted customers.

## 4. Combined risk score and portfolio selection

CV and HHI use different scales, so each is divided by its value for the revenue-only top-100 baseline. With variability weight $w$ and concentration weight $1-w$:

$$Score_P = 100\left[w\frac{CV_P}{CV_{baseline}} + (1-w)\frac{HHI_P}{HHI_{baseline}}\right]$$

The delivered model uses $w=0.5$, so stability and customer concentration receive equal weight. The revenue-only baseline has a score of 100. A score below 100 indicates lower combined in-sample risk than that baseline. The score is **not a probability**.

For each retention level, the optimizer chooses exactly 100 merchants, respects the expected-revenue floor, and minimizes the combined score. `portfolio_score_lower_bound`, `portfolio_optimality_gap`, and `portfolio_certified` describe optimization evidence; certification concerns the numerical solution tolerance, not future performance.

## 5. Revenue and customer-concentration correlation matrix

The dashboard repeats portfolio metrics on each merchant row. The next cell first keeps one row per portfolio and then calculates the **Pearson correlation across the 30 delivered portfolios** (six dashboard scopes × five retention levels). This avoids incorrectly weighting a portfolio 100 times. Correlation is descriptive only and does not imply causation.

In [2]:
portfolio_rows = pd.concat(
    [
        frame.drop_duplicates("portfolio_id").assign(dashboard=label)
        for label, frame in dashboards.items()
    ],
    ignore_index=True,
)

correlation_data = portfolio_rows[[
    "portfolio_expected_monthly_adjusted_fees",
    "portfolio_customer_hhi",
]].rename(
    columns={
        "portfolio_expected_monthly_adjusted_fees": "Expected monthly revenue",
        "portfolio_customer_hhi": "Customer concentration (HHI)",
    }
)

correlation_matrix = correlation_data.corr(method="pearson")
display(correlation_matrix.round(3))

,Expected monthly revenue,Customer concentration (HHI)
Expected monthly revenue,1.000000,-0.607000
Customer concentration (HHI),-0.607000,1.000000


## 6. Required output views: global 100 and top 10 per segment

The default below remains **α = 100%**, the adjusted-revenue baseline. Its complete global list and each segment's first ten ranks are displayed. The dashboard opens the already delivered **α = 99%** portfolio; a second, explicitly labelled view below shows that same saved choice. These are two existing settings, not two newly fitted models. The later comparison uses 100% versus 99% to explain the return–risk trade-off. No retention setting is inferred to be universally best from its name or default.

Within every saved portfolio, ranks order merchants by adjusted monthly fees. **A segment's top 10 here means ranks 1–10 of its existing 100-merchant portfolio.** This display does not rerun the optimizer with a size of ten and does not claim that those ten form a separately optimized ten-merchant portfolio. The five groups and all saved selections remain unchanged.

Change `SELECTED_RETENTION` to another saved setting to inspect it. All tables retain the exported ranks; the global display is explicitly untruncated. These outputs still describe selected portfolios, rather than claiming an independent risk ranking of every merchant in the source data.


In [3]:
SELECTED_RETENTION = 1.0
DASHBOARD_RETENTION = 0.99

RANKING_COLUMNS = [
    "rank_by_expected_fees",
    "merchant_abn",
    "merchant_name",
    "merchant_category",
    "merchant_expected_monthly_adjusted_fees",
]

def saved_ranking(frame, retention):
    """Read the existing portfolio in its supplied rank order; never reselect."""
    selected = frame.loc[np.isclose(frame["retention"], retention)].copy()
    if len(selected) != 100 or selected.merchant_abn.nunique() != 100:
        raise ValueError(f"Expected 100 unique merchants at retention={retention}")
    ordered = selected.sort_values("rank_by_expected_fees")
    if not np.array_equal(ordered.rank_by_expected_fees.to_numpy(), np.arange(1, 101)):
        raise ValueError("The saved portfolio must have ranks 1–100.")
    return ordered[RANKING_COLUMNS]

print(f"Complete global list — retention {SELECTED_RETENTION:.1%}")
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(saved_ranking(dashboards["Overall"], SELECTED_RETENTION))


Complete global list — retention 100.0%


,rank_by_expected_fees,merchant_abn,merchant_name,merchant_category,merchant_expected_monthly_adjusted_fees
0,1,32361057556,Orci In Consequat Corporation,"gift, card, novelty, and souvenir shops","29,658.603238"
1,2,48534649627,Dignissim Maecenas Foundation,"opticians, optical goods, and eyeglasses","29,586.592630"
2,3,79827781481,Amet Risus Inc.,"furniture, home furnishings and equipment shops, and manufacturers, except appliances","29,337.700585"
3,4,86578477987,Leo In Consulting,"watch, clock, and jewelry repair shops","29,283.455366"
4,5,45629217853,Lacus Consulting,"gift, card, novelty, and souvenir shops","27,975.322625"
5,6,38700038932,Etiam Bibendum Industries,tent and awning shops,"27,546.711107"
6,7,21439773999,Mauris Non Institute,"cable, satellite, and other pay television and radio services","27,370.211774"
7,8,96680767841,Ornare Limited,motor vehicle supplies and new parts,"27,364.493820"
8,9,64403598239,Lobortis Ultrices Company,"music shops - musical instruments, pianos, and sheet music","26,659.867343"
9,10,63123845164,Odio Phasellus Institute,artist supply and craft shops,"26,313.522216"


In [4]:
for label, frame in dashboards.items():
    if label == "Overall":
        continue
    print(f"Top 10 in the saved {label} portfolio — retention {SELECTED_RETENTION:.1%}")
    with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
        display(saved_ranking(frame, SELECTED_RETENTION).head(10))

if not np.isclose(SELECTED_RETENTION, DASHBOARD_RETENTION):
    display(Markdown("**Existing dashboard view — α = 99%.** The following tables read the already delivered portfolios without changing the notebook's 100% baseline default."))
    print(f"Complete global list — dashboard retention {DASHBOARD_RETENTION:.1%}")
    with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
        display(saved_ranking(dashboards["Overall"], DASHBOARD_RETENTION))
    for label, frame in dashboards.items():
        if label == "Overall":
            continue
        print(f"Top 10 in the saved {label} portfolio — dashboard retention {DASHBOARD_RETENTION:.1%}")
        with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
            display(saved_ranking(frame, DASHBOARD_RETENTION).head(10))


Top 10 in the saved Arts, Crafts, Gifts & Entertainment portfolio — retention 100.0%


,rank_by_expected_fees,merchant_abn,merchant_name,merchant_category,merchant_expected_monthly_adjusted_fees
0,1,32361057556,Orci In Consequat Corporation,"gift, card, novelty, and souvenir shops","29,658.603238"
1,2,45629217853,Lacus Consulting,"gift, card, novelty, and souvenir shops","27,975.322625"
2,3,64403598239,Lobortis Ultrices Company,"music shops - musical instruments, pianos, and sheet music","26,659.867343"
3,4,63123845164,Odio Phasellus Institute,artist supply and craft shops,"26,313.522216"
4,5,94493496784,Dictum Phasellus In Institute,"gift, card, novelty, and souvenir shops","24,514.671545"
5,6,79417999332,Phasellus At Company,"gift, card, novelty, and souvenir shops","21,515.146761"
6,7,40515428545,Elit Sed Consequat Associates,artist supply and craft shops,"21,427.127966"
7,8,67978471888,Magna Malesuada Corp.,artist supply and craft shops,"21,031.950080"
8,9,98166254020,Magna Sed Industries,art dealers and galleries,"19,548.487080"
9,10,98973094975,Ornare Fusce Inc.,"hobby, toy and game shops","19,406.483781"


Top 10 in the saved Automotive, Outdoor & Recreational Equipment portfolio — retention 100.0%


,rank_by_expected_fees,merchant_abn,merchant_name,merchant_category,merchant_expected_monthly_adjusted_fees
0,1,38700038932,Etiam Bibendum Industries,tent and awning shops,"27,546.711107"
1,2,96680767841,Ornare Limited,motor vehicle supplies and new parts,"27,364.493820"
2,3,89726005175,Est Nunc Consulting,tent and awning shops,"25,508.229026"
3,4,49891706470,Non Vestibulum Industries,tent and awning shops,"19,834.214599"
4,5,80551528183,Ac Ipsum LLC,tent and awning shops,"19,554.967058"
5,6,90568944804,Diam Eu Dolor LLC,tent and awning shops,"18,257.676807"
6,7,13514558491,Magna Praesent PC,motor vehicle supplies and new parts,"18,080.994982"
7,8,75454398468,Tempus Non Lacinia Corporation,tent and awning shops,"18,023.417504"
8,9,49549583265,Luctus Et Incorporated,tent and awning shops,"17,953.882212"
9,10,31385641294,Semper Auctor PC,motor vehicle supplies and new parts,"16,243.903839"


Top 10 in the saved Fashion, Jewellery & Personal Care portfolio — retention 100.0%


...

,rank_by_expected_fees,merchant_abn,merchant_name,merchant_category,merchant_expected_monthly_adjusted_fees
200,1,48534649627,Dignissim Maecenas Foundation,"opticians, optical goods, and eyeglasses","29,586.592630"
201,2,86578477987,Leo In Consulting,"watch, clock, and jewelry repair shops","29,283.455366"
202,3,49322182190,Gravida Mauris Incorporated,"watch, clock, and jewelry repair shops","23,711.569787"
203,4,93558142492,Dolor Quisque Inc.,shoe shops,"14,225.877694"
204,5,11439466003,Blandit At LLC,shoe shops,"13,174.395798"
205,6,19492220327,Commodo Ipsum Industries,"jewelry, watch, clock, and silverware shops","12,903.479267"
206,7,95574756848,At Pede Inc.,"opticians, optical goods, and eyeglasses","11,836.219852"
207,8,99976658299,Sociosqu Corp.,shoe shops,"10,809.156607"
208,9,62224020443,Hendrerit A Corporation,"watch, clock, and jewelry repair shops","10,723.302486"
209,10,46804135891,Suspendisse Dui Corporation,"opticians, optical goods, and eyeglasses","9,833.162584"


Top 10 in the saved Home, Garden & General Supplies portfolio — dashboard retention 99.0%


,rank_by_expected_fees,merchant_abn,merchant_name,merchant_category,merchant_expected_monthly_adjusted_fees
200,1,79827781481,Amet Risus Inc.,"furniture, home furnishings and equipment shops, and manufacturers, except appliances","29,337.700585"
201,2,76767266140,Phasellus At Limited,"furniture, home furnishings and equipment shops, and manufacturers, except appliances","20,590.482745"
202,3,43186523025,Lorem Ipsum Sodales Industries,"florists supplies, nursery stock, and flowers","19,281.527463"
203,4,49212265466,Auctor Company,"florists supplies, nursery stock, and flowers","17,303.389075"
204,5,57757792876,Pretium Et LLC,"stationery, office supplies and printing and writing paper","16,382.202827"
205,6,21772962346,Purus Gravida Sagittis Ltd,"florists supplies, nursery stock, and flowers","16,237.197761"
206,7,42355028515,Eu Inc.,"lawn and garden supply outlets, including nurseries","15,579.215700"
207,8,76314317957,Semper Corp.,"florists supplies, nursery stock, and flowers","13,982.689298"
208,9,38090089066,Interdum Feugiat Sed Inc.,"furniture, home furnishings and equipment shops, and manufacturers, except appliances","13,686.511743"
209,10,32709545238,Tempor Est Foundation,"stationery, office supplies and printing and writing paper","13,672.847217"


Top 10 in the saved Technology, Digital & Communications portfolio — dashboard retention 99.0%


,rank_by_expected_fees,merchant_abn,merchant_name,merchant_category,merchant_expected_monthly_adjusted_fees
200,1,21439773999,Mauris Non Institute,"cable, satellite, and other pay television and radio services","27,370.211774"
201,2,82368304209,Nec Incorporated,telecom,"23,689.830228"
202,3,35909341340,Arcu Sed Eu Incorporated,"computer programming, data processing, and integrated systems design services","21,620.409540"
203,4,72472909171,Nullam Consulting,"digital goods: books, movies, music","21,576.427140"
204,5,45433476494,Adipiscing Elit Foundation,"computers, computer peripheral equipment, and software","19,648.431615"
205,6,58454491168,Diam At Foundation,"computer programming, data processing, and integrated systems design services","19,063.362015"
206,7,94690988633,Eu Placerat LLC,"computers, computer peripheral equipment, and software","18,445.682465"
207,8,49505931725,Suspendisse Ac Associates,"digital goods: books, movies, music","16,069.315033"
208,9,67400260923,Eleifend PC,"computer programming, data processing, and integrated systems design services","15,933.900373"
209,10,80518954462,Neque Sed Dictum Incorporated,"computers, computer peripheral equipment, and software","14,699.234447"


## Interpretation notes

- Revenue is an expectation based on historical averages, not a guaranteed forecast.
- CV and HHI describe different risks: weekly cash-flow instability and dependence on shared customers.
- Relaxing the revenue-retention floor gives the optimizer more freedom to reduce risk, but actual achieved revenue retention may be above the minimum.
- Portfolio-level fields are repeated on all 100 merchant rows. Use one row per `portfolio_id` for portfolio analysis; only `merchant_expected_monthly_adjusted_fees` should be summed within a portfolio.
- The rankings order merchants inside an already selected portfolio by expected fees. They are not individual merchant risk rankings.

## 7. Presentation analysis: definitions and source checks

The following sections extend the original summary without rerunning optimisation. **Expected monthly adjusted return** means the final export's take-rate-adjusted monthly return **after consumer-fraud adjustment** (`merchant_expected_monthly_adjusted_fees`). The same definition is used for merchant and portfolio comparisons. It is not net profit. Curated `expected_revenue` is a different, fraud-adjusted historical total and is deliberately not substituted.

Here **α denotes the return-retention constraint**, not the covariance shrinkage parameter used earlier. The reference comparison is α = 100% versus 99%; other tables discover available alpha values from the files. A single snapshot, risk window, portfolio size and weight configuration is required so unrelated runs cannot be accidentally pooled.

Additional sources: `handoff/fraud_adjusted/input_snapshot` for eligibility, customer exposures and weekly histories; `tables/curated/merchants` for merchant attributes; and the **delivered per-segment final exports** for category-to-segment membership. These exports are the source for describing the frozen selections. A newly regenerated exploratory `category_segments.parquet` is not used to relabel an existing selection. Updating these inputs and rerunning the notebook recalculates all results. Inputs must belong to the same model run; a changed return definition requires corresponding model exports.

In [5]:
from IPython.display import Markdown

BASELINE_ALPHA = 1.0
COMPARISON_ALPHA = 0.99
SEGMENT_ORDER = [
    "Technology, Digital & Communications",
    "Home, Garden & General Supplies",
    "Fashion, Jewellery & Personal Care",
    "Arts, Crafts, Gifts & Entertainment",
    "Automotive, Outdoor & Recreational Equipment",
]
SNAPSHOT_DIR = PROJECT_ROOT / "handoff" / "fraud_adjusted" / "input_snapshot"
GLOBAL = dashboards["Overall"].copy()
all_rankings = pd.concat([f.assign(scope=s) for s, f in dashboards.items()], ignore_index=True)
for column in ["as_of", "window_start", "window_end", "portfolio_size", "variability_weight", "concentration_weight"]:
    if all_rankings[column].nunique(dropna=False) != 1:
        raise ValueError(f"Select a single configuration before summarising: {column}")
for scope, frame in all_rankings.groupby("scope"):
    if frame.duplicated(["portfolio_id", "merchant_abn"]).any():
        raise ValueError(f"Duplicate merchant in portfolio: {scope}")
    if frame.groupby("retention")["portfolio_id"].nunique().ne(1).any():
        raise ValueError(f"Multiple portfolios at the same alpha: {scope}")
    for ident, rows in frame.groupby("portfolio_id"):
        if len(rows) != int(rows.portfolio_size.iloc[0]):
            raise ValueError(f"Incorrect portfolio size: {scope}, {ident}")
        if any(rows[c].nunique(dropna=False) != 1 for c in rows if c.startswith("portfolio_")):
            raise ValueError(f"Inconsistent portfolio metrics: {ident}")
        if not np.isclose(rows.merchant_expected_monthly_adjusted_fees.sum(), rows.portfolio_expected_monthly_adjusted_fees.iloc[0]):
            raise ValueError(f"Merchant and portfolio return totals disagree: {ident}")

metrics = GLOBAL.drop_duplicates("portfolio_id").set_index("retention").sort_index(ascending=False)
def alpha_key(value):
    matches = metrics.index[np.isclose(metrics.index, value)]
    if len(matches) != 1:
        raise ValueError(f"Required comparison alpha {value:g} is unavailable; update COMPARISON_ALPHA/BASELINE_ALPHA")
    return matches[0]
base_alpha, compare_alpha = alpha_key(BASELINE_ALPHA), alpha_key(COMPARISON_ALPHA)
base_metrics, compare_metrics = metrics.loc[base_alpha], metrics.loc[compare_alpha]

# Describe the fixed delivered scopes, independently of exploratory mapping files.
segment_map = pd.concat([
    pd.DataFrame({"categories": frame.merchant_category.unique(), "segments": scope})
    for scope, frame in dashboards.items() if scope != "Overall"
], ignore_index=True)
print("Segment-membership source: delivered fraud-adjusted per-segment final exports.")
if segment_map.categories.duplicated().any() or set(segment_map.segments) != set(SEGMENT_ORDER):
    raise ValueError("Invalid category-to-segment mapping")
category_to_segment = segment_map.set_index("categories").segments
all_rankings["segment"] = all_rankings.merchant_category.map(category_to_segment)
if all_rankings.segment.isna().any():
    raise ValueError("Unmapped selected merchant categories")
for scope, rows in all_rankings.groupby("scope"):
    if scope != "Overall" and not rows.segment.eq(scope).all():
        raise ValueError(f"Merchant outside its declared segment: {scope}")

Segment-membership source: delivered fraud-adjusted per-segment final exports.


## 8. Global return–risk trade-offs

Return sacrificed is the baseline monthly adjusted return minus the selected portfolio's monthly adjusted return. The combined risk score is an index with baseline 100, not a loss probability or monetary saving. Solver certification and gaps are retained because “best found” portfolios are not necessarily certified optimal.

In [6]:
tradeoffs = pd.DataFrame({
    "Expected monthly return (AUD)": metrics.portfolio_expected_monthly_adjusted_fees,
    "Monthly return sacrificed vs baseline (AUD)": base_metrics.portfolio_expected_monthly_adjusted_fees - metrics.portfolio_expected_monthly_adjusted_fees,
    "Combined risk index": metrics.portfolio_risk_score,
    "Certified": metrics.portfolio_certified,
    "Gap upper bound (%)": metrics.portfolio_optimality_gap * 100,
})
tradeoffs.index = [f"α = {a*100:g}%" for a in tradeoffs.index]
display(tradeoffs.style.format({
    "Expected monthly return (AUD)": "A${:,.0f}",
    "Monthly return sacrificed vs baseline (AUD)": "A${:,.0f}",
    "Combined risk index": "{:.2f}", "Gap upper bound (%)": "{:.3f}",
}))

,Expected monthly return (AUD),Monthly return sacrificed vs baseline (AUD),Combined risk index,Certified,Gap upper bound (%)
α = 100%,"A$1,551,710",A$0,100.00,True,0.000
α = 99.5%,"A$1,544,101","A$7,609",99.06,True,0.043
α = 99%,"A$1,536,393","A$15,317",98.60,True,0.048
α = 98%,"A$1,520,698","A$31,012",97.82,True,0.096
α = 95%,"A$1,474,509","A$77,201",96.17,False,0.712


## 9. Merchant substitutions and change in portfolio metrics

“Removed” and “added” describe the change from the baseline to the comparison alpha. Changes in portfolio risk cannot be attributed to individual merchants independently: covariance and shared customers are evaluated jointly. Risk-index points and relative percentage reductions are reported separately.
The first table also reports all-history unique consumers, derived from the processed curated transactions and the existing user-to-consumer identity bridge. It is recalculated for the changed merchants on each run, so updated inputs cannot leave stale counts.


In [7]:
base_rows = GLOBAL.loc[np.isclose(GLOBAL.retention, base_alpha)].set_index("merchant_abn")
comparison_rows = GLOBAL.loc[np.isclose(GLOBAL.retention, compare_alpha)].set_index("merchant_abn")
removed_ids = set(base_rows.index) - set(comparison_rows.index)
added_ids = set(comparison_rows.index) - set(base_rows.index)
changes = pd.concat([
    base_rows.loc[sorted(removed_ids)].assign(Change="Removed"),
    comparison_rows.loc[sorted(added_ids)].assign(Change="Added"),
]).reset_index()
# Count distinct consumers over ALL stored curated transactions, not the
# 26-week customer-exposure snapshot used for portfolio risk.
import pyarrow as pa
import pyarrow.dataset as ds
consumer_bridge = pd.read_parquet(
    PROJECT_ROOT / "tables/landing/project-2-bnpl-tables-part1/consumer_user_details.parquet",
    columns=["user_id", "consumer_id"],
).drop_duplicates()
if (consumer_bridge.isna().any().any() or consumer_bridge.user_id.duplicated().any()
        or consumer_bridge.consumer_id.duplicated().any()):
    raise ValueError("Expected a complete one-to-one user/consumer identity bridge")
user_to_consumer = consumer_bridge.set_index("user_id").consumer_id.to_dict()
transaction_history = ds.dataset(PROJECT_ROOT / "tables/curated/transactions", format="parquet", exclude_invalid_files=True)
changed_abns = set(changes.merchant_abn)
consumer_sets = {abn: set() for abn in changed_abns}
coverage_start, coverage_end = None, None
if changed_abns:
    abn_type = transaction_history.schema.field("merchant_abn").type
    filter_ids = pa.array([int(a) if pa.types.is_integer(abn_type) else a for a in changed_abns], type=abn_type)
    scanner = transaction_history.scanner(
        columns=["merchant_abn", "user_id", "order_datetime"],
        filter=ds.field("merchant_abn").isin(filter_ids), batch_size=65536,
    )
    for batch in scanner.to_batches():
        for abn, user, order_date in zip(*(batch.column(k).to_pylist() for k in ["merchant_abn", "user_id", "order_datetime"])):
            if user not in user_to_consumer:
                raise ValueError(f"Transaction user has no consumer mapping: {user}")
            consumer_sets[str(abn)].add(user_to_consumer[user])
            if order_date is not None:
                coverage_start = order_date if coverage_start is None else min(coverage_start, order_date)
                coverage_end = order_date if coverage_end is None else max(coverage_end, order_date)
consumer_counts_all_history = pd.Series({abn: len(consumers) for abn, consumers in consumer_sets.items()}, dtype="int64")
changes["Unique consumers (all history)"] = changes.merchant_abn.map(consumer_counts_all_history).astype("int64")
changes = changes[["Change", "merchant_abn", "merchant_name", "merchant_category", "Unique consumers (all history)", "merchant_expected_monthly_adjusted_fees"]].rename(columns={"merchant_expected_monthly_adjusted_fees": "Expected monthly return (AUD)"})
display(changes.style.format({"Expected monthly return (AUD)": "A${:,.2f}", "Unique consumers (all history)": "{:,.0f}"}))
if coverage_start is not None:
    display(Markdown(f"Consumer counts include every distinct mapped consumer in the stored curated transaction history, including repeat buyers only once. Transactions for these merchants span **{coverage_start}–{coverage_end}**. This is a descriptive all-history count, not the 26-week risk-window count or an as-of-snapshot measure."))
comparison_specs = {
    "Monthly adjusted return (AUD)": "portfolio_expected_monthly_adjusted_fees",
    "Combined risk index": "portfolio_risk_score",
    "Weekly return CV": "portfolio_variability_cv",
    "Consumer concentration (HHI)": "portfolio_customer_hhi",
}
comparison = pd.DataFrame([
    {"Metric": label, "Baseline": float(base_metrics[key]), "Comparison": float(compare_metrics[key]),
     "Change (comparison − baseline)": float(compare_metrics[key] - base_metrics[key]),
     "Relative reduction (%)": (1-float(compare_metrics[key])/float(base_metrics[key]))*100 if base_metrics[key] else np.nan}
    for label, key in comparison_specs.items()
]).set_index("Metric")
display(comparison)
return_lost = float(base_metrics.portfolio_expected_monthly_adjusted_fees - compare_metrics.portfolio_expected_monthly_adjusted_fees)
risk_reduction = float(base_metrics.portfolio_risk_score - compare_metrics.portfolio_risk_score)
display(Markdown(f"At **α = {compare_alpha:.3%}** versus **{base_alpha:.3%}**, **{len(removed_ids)}** merchants are removed and **{len(added_ids)}** added. Monthly adjusted return changes by **A${-return_lost:,.2f}**; the combined risk index changes by **{-risk_reduction:.3f} points**. These are in-sample trade-offs, not demonstrated reductions in realised losses."))
assert np.isclose(base_rows.loc[sorted(removed_ids), "merchant_expected_monthly_adjusted_fees"].sum() - comparison_rows.loc[sorted(added_ids), "merchant_expected_monthly_adjusted_fees"].sum(), return_lost)

,Change,merchant_abn,merchant_name,merchant_category,Unique consumers (all history),Expected monthly return (AUD)
0,Removed,12771097467,At Pretium Corp.,motor vehicle supplies and new parts,"1,297","A$14,415.71"
1,Removed,19492220327,Commodo Ipsum Industries,"jewelry, watch, clock, and silverware shops",810,"A$12,903.48"
2,Removed,29639699851,Sodales Elit Erat Corporation,"lawn and garden supply outlets, including nurseries","7,765","A$7,410.38"
3,Removed,35809331583,Euismod In LLC,"furniture, home furnishings and equipment shops, and manufacturers, except appliances","1,752","A$8,595.77"
4,Added,39649557865,Arcu Morbi Institute,artist supply and craft shops,"14,462","A$6,788.23"
5,Added,61080648284,Lobortis Ultrices LLC,artist supply and craft shops,"3,657","A$7,135.70"
6,Added,70009327857,Torquent Per Inc.,"lawn and garden supply outlets, including nurseries","4,454","A$7,376.50"
7,Added,98269572896,Eget Laoreet Posuere PC,"furniture, home furnishings and equipment shops, and manufacturers, except appliances","20,484","A$6,707.65"


Consumer counts include every distinct mapped consumer in the stored curated transaction history, including repeat buyers only once. Transactions for these merchants span **2021-02-28–2022-10-26**. This is a descriptive all-history count, not the 26-week risk-window count or an as-of-snapshot measure.

,Baseline,Comparison,Change (comparison − baseline),Relative reduction (%)
Metric,,,,
Monthly adjusted return (AUD),"1,551,710.015613","1,536,392.767695","-15,317.247918",0.987121
Combined risk index,100.000000,98.601015,-1.398985,1.398985
Weekly return CV,0.031376,0.030883,-0.000493,1.570806
Consumer concentration (HHI),0.000046,0.000045,-0.000001,1.227164


At **α = 99.000%** versus **100.000%**, **4** merchants are removed and **4** added. Monthly adjusted return changes by **A$-15,317.25**; the combined risk index changes by **-1.399 points**. These are in-sample trade-offs, not demonstrated reductions in realised losses.

## 10. Segment composition of every ranking

Percentages represent **merchant counts**, not return weights. All five segments are displayed, including zero counts. Global counts equal percentages when portfolio size is 100; both are calculated independently so this remains valid if portfolio sizes change. Each per-segment portfolio should contain 100% merchants from its own segment.

In [8]:
segment_counts_by_scope = {}
segment_percentages_by_scope = {}
for scope in dashboards:
    rows = all_rankings.loc[all_rankings.scope.eq(scope)]
    counts = pd.crosstab(rows.segment, rows.retention).reindex(index=SEGMENT_ORDER, columns=sorted(rows.retention.unique()), fill_value=0)
    percentages = counts.div(counts.sum(axis=0), axis=1) * 100
    assert np.allclose(percentages.sum(axis=0), 100)
    segment_counts_by_scope[scope] = counts
    segment_percentages_by_scope[scope] = percentages
    display(Markdown(f"**{scope}: merchant proportions**"))
    display(percentages.rename(columns=lambda a: f"α = {a*100:g}%").style.format("{:.1f}%"))
display(Markdown("**Global merchant counts (presentation table)**"))
display(segment_counts_by_scope["Overall"].rename(columns=lambda a: f"α = {a*100:g}%"))

**Overall: merchant proportions**

retention,α = 95%,α = 98%,α = 99%,α = 99.5%,α = 100%
segment,,,,,
"Technology, Digital & Communications",17.0%,19.0%,17.0%,17.0%,17.0%
"Home, Garden & General Supplies",17.0%,16.0%,19.0%,18.0%,19.0%
"Fashion, Jewellery & Personal Care",15.0%,14.0%,14.0%,14.0%,15.0%
"Arts, Crafts, Gifts & Entertainment",32.0%,30.0%,29.0%,29.0%,27.0%
"Automotive, Outdoor & Recreational Equipment",19.0%,21.0%,21.0%,22.0%,22.0%


**Arts, Crafts, Gifts & Entertainment: merchant proportions**

retention,α = 95%,α = 98%,α = 99%,α = 99.5%,α = 100%
segment,,,,,
"Technology, Digital & Communications",0.0%,0.0%,0.0%,0.0%,0.0%
"Home, Garden & General Supplies",0.0%,0.0%,0.0%,0.0%,0.0%
"Fashion, Jewellery & Personal Care",0.0%,0.0%,0.0%,0.0%,0.0%
"Arts, Crafts, Gifts & Entertainment",100.0%,100.0%,100.0%,100.0%,100.0%
"Automotive, Outdoor & Recreational Equipment",0.0%,0.0%,0.0%,0.0%,0.0%


**Automotive, Outdoor & Recreational Equipment: merchant proportions**

...

retention,α = 95%,α = 98%,α = 99%,α = 99.5%,α = 100%
segment,,,,,
"Technology, Digital & Communications",0.0%,0.0%,0.0%,0.0%,0.0%
"Home, Garden & General Supplies",100.0%,100.0%,100.0%,100.0%,100.0%
"Fashion, Jewellery & Personal Care",0.0%,0.0%,0.0%,0.0%,0.0%
"Arts, Crafts, Gifts & Entertainment",0.0%,0.0%,0.0%,0.0%,0.0%
"Automotive, Outdoor & Recreational Equipment",0.0%,0.0%,0.0%,0.0%,0.0%


**Technology, Digital & Communications: merchant proportions**

retention,α = 95%,α = 98%,α = 99%,α = 99.5%,α = 100%
segment,,,,,
"Technology, Digital & Communications",100.0%,100.0%,100.0%,100.0%,100.0%
"Home, Garden & General Supplies",0.0%,0.0%,0.0%,0.0%,0.0%
"Fashion, Jewellery & Personal Care",0.0%,0.0%,0.0%,0.0%,0.0%
"Arts, Crafts, Gifts & Entertainment",0.0%,0.0%,0.0%,0.0%,0.0%
"Automotive, Outdoor & Recreational Equipment",0.0%,0.0%,0.0%,0.0%,0.0%


**Global merchant counts (presentation table)**

retention,α = 95%,α = 98%,α = 99%,α = 99.5%,α = 100%
segment,,,,,
"Technology, Digital & Communications",17,19,17,17,17
"Home, Garden & General Supplies",17,16,19,18,19
"Fashion, Jewellery & Personal Care",15,14,14,14,15
"Arts, Crafts, Gifts & Entertainment",32,30,29,29,27
"Automotive, Outdoor & Recreational Equipment",19,21,21,22,22


## 11. Core merchants and their contribution to return

Always selected = present at every available global alpha. Sometimes selected = present at least once but not at every alpha. Never selected = eligible in the frozen input snapshot but absent from all global portfolios. Ineligible merchants are excluded from this comparison. These are selection-frequency groups, not automatic acceptance/rejection labels.

In [9]:
snapshot_merchants = pd.read_parquet(SNAPSHOT_DIR / "merchants.parquet")
snapshot_merchants["merchant_abn"] = snapshot_merchants.merchant_abn.astype("string")
if snapshot_merchants.merchant_abn.duplicated().any():
    raise ValueError("Duplicate snapshot merchant ABN")
if not snapshot_merchants.eligible.isin([True, False]).all():
    raise ValueError("Snapshot eligibility must be boolean")
eligible = snapshot_merchants.loc[snapshot_merchants.eligible.eq(True)].set_index("merchant_abn").copy()
if not set(GLOBAL.merchant_abn).issubset(eligible.index):
    raise ValueError("Final ranking includes merchant absent from eligible snapshot")
snapshot_returns = GLOBAL.merchant_abn.map(eligible.expected_monthly_fees)
if not np.allclose(snapshot_returns, GLOBAL.merchant_expected_monthly_adjusted_fees):
    raise ValueError("Final returns disagree with frozen snapshot")
selection_frequency = GLOBAL.groupby("merchant_abn").retention.nunique()
n_alphas = GLOBAL.retention.nunique()
eligible["selection_count"] = selection_frequency.reindex(eligible.index, fill_value=0)
eligible["Selection group"] = np.select([eligible.selection_count.eq(n_alphas), eligible.selection_count.gt(0)], ["Always selected", "Sometimes selected"], default="Never selected")
core = eligible.loc[eligible["Selection group"].eq("Always selected")].copy()
selected_union = int(eligible.selection_count.gt(0).sum())
core_return = comparison_rows.loc[comparison_rows.index.isin(core.index), "merchant_expected_monthly_adjusted_fees"].sum()
core_return_share = core_return / compare_metrics.portfolio_expected_monthly_adjusted_fees
alpha_memberships = GLOBAL.groupby("merchant_abn").retention.apply(lambda v: ", ".join(f"{a*100:g}%" for a in sorted(v.unique(), reverse=True)))
core["Selected at α"] = alpha_memberships.reindex(core.index)
display(Markdown(f"**{selected_union:,}** of **{len(eligible):,}** eligible merchants occur in at least one global portfolio; **{len(core):,}** occur in all **{n_alphas}**. The core contributes **A${core_return:,.2f}**, or **{core_return_share:.2%}**, of monthly adjusted return at **α = {compare_alpha*100:g}%**."))
display(core.sort_values("expected_monthly_fees", ascending=False)[["name", "category", "expected_monthly_fees", "Selected at α"]].rename(columns={"expected_monthly_fees": "Expected monthly return (AUD)"}).style.format({"Expected monthly return (AUD)": "A${:,.2f}"}))
core_segments = core.category.map(category_to_segment).value_counts().reindex(SEGMENT_ORDER, fill_value=0)
display(core_segments.rename("Core merchant count").to_frame())

**113** of **3,897** eligible merchants occur in at least one global portfolio; **90** occur in all **5**. The core contributes **A$1,428,007.06**, or **92.95%**, of monthly adjusted return at **α = 99%**.

,name,category,Expected monthly return (AUD),Selected at α
merchant_abn,,,,
32361057556,Orci In Consequat Corporation,"gift, card, novelty, and souvenir shops","A$29,658.60","100%, 99.5%, 99%, 98%, 95%"
48534649627,Dignissim Maecenas Foundation,"opticians, optical goods, and eyeglasses","A$29,586.59","100%, 99.5%, 99%, 98%, 95%"
79827781481,Amet Risus Inc.,"furniture, home furnishings and equipment shops, and manufacturers, except appliances","A$29,337.70","100%, 99.5%, 99%, 98%, 95%"
86578477987,Leo In Consulting,"watch, clock, and jewelry repair shops","A$29,283.46","100%, 99.5%, 99%, 98%, 95%"
45629217853,Lacus Consulting,"gift, card, novelty, and souvenir shops","A$27,975.32","100%, 99.5%, 99%, 98%, 95%"
38700038932,Etiam Bibendum Industries,tent and awning shops,"A$27,546.71","100%, 99.5%, 99%, 98%, 95%"
21439773999,Mauris Non Institute,"cable, satellite, and other pay television and radio services","A$27,370.21","100%, 99.5%, 99%, 98%, 95%"
96680767841,Ornare Limited,motor vehicle supplies and new parts,"A$27,364.49","100%, 99.5%, 99%, 98%, 95%"
64403598239,Lobortis Ultrices Company,"music shops - musical instruments, pianos, and sheet music","A$26,659.87","100%, 99.5%, 99%, 98%, 95%"


,Core merchant count
category,
"Technology, Digital & Communications",16
"Home, Garden & General Supplies",15
"Fashion, Jewellery & Personal Care",14
"Arts, Crafts, Gifts & Entertainment",26
"Automotive, Outdoor & Recreational Equipment",19


## 12. What distinguishes always-, sometimes- and never-selected merchants?

All statistics below are **merchant-level medians**, not pooled customer statistics. Customer counts and largest-customer shares use customer–merchant return over the snapshot's risk window. Weekly variability is individual sample standard deviation (`ddof=1`) divided by mean weekly return, including zero weeks; it is **not** the shrunk portfolio CV. Largest consumer's share is maximum customer return divided by the merchant's total customer return, distinct from portfolio HHI.

Take rates and average fraud scores are stored in percentage points. Curated average transaction values and fraud scores cover the stored history, which may extend beyond the frozen model snapshot; they are descriptive attributes, not evidence of out-of-sample prediction. Fraud scores are not realised fraud rates. Return, weekly histories and consumer exposures in this table all use the reconstructed consumer-fraud-adjusted basis.
The adjusted snapshot retains the internal names `gross_fees` and `expected_monthly_fees`; in this folder they contain adjusted amounts. Curated descriptive fraud averages remain from the separate curated history and may differ from the reconstructed scores used in optimisation.


In [10]:
weekly = pd.read_parquet(SNAPSHOT_DIR / "weekly_fees.parquet")
customers = pd.read_parquet(SNAPSHOT_DIR / "customer_fees.parquet")
for frame in [weekly, customers]:
    frame["merchant_abn"] = frame.merchant_abn.astype("string")
if weekly.duplicated(["merchant_abn", "week_end"]).any() or customers.duplicated(["merchant_abn", "consumer_id"]).any():
    raise ValueError("Duplicate weekly or customer–merchant rows")
weekly["week_end"] = pd.to_datetime(weekly.week_end)
start, end = pd.Timestamp(base_metrics.window_start), pd.Timestamp(base_metrics.window_end)
if not weekly.week_end.between(start, end).all():
    raise ValueError("Weekly histories extend outside portfolio risk window")
week_count = len(pd.date_range(start, end, freq="W-SUN"))
if not weekly.groupby("merchant_abn").size().reindex(eligible.index, fill_value=0).eq(week_count).all():
    raise ValueError("Missing weekly rows: genuine zero weeks must be present")
weekly_stats = weekly.groupby("merchant_abn").gross_fees.agg(["mean", "std"])
weekly_stats["weekly_cv"] = weekly_stats["std"].div(weekly_stats["mean"].where(weekly_stats["mean"].gt(0)))
customer_stats = customers.groupby("merchant_abn").agg(customers=("consumer_id", "nunique"), total=("gross_fees", "sum"), largest=("gross_fees", "max"))
customer_stats["largest_share"] = customer_stats.largest.div(customer_stats.total.where(customer_stats.total.gt(0)))
curated = pd.read_parquet(PROJECT_ROOT / "tables/curated/merchants", columns=["merchant_abn", "take_rate", "avg_value_per_transaction", "avg_consumer_fraud_prob"])
curated["merchant_abn"] = curated.merchant_abn.astype("string")
if curated.merchant_abn.duplicated().any() or not set(eligible.index).issubset(curated.merchant_abn):
    raise ValueError("Curated merchant join is incomplete or duplicated")
profiles = eligible[["name", "Selection group", "selection_count", "expected_monthly_fees"]].join(weekly_stats[["weekly_cv"]]).join(customer_stats[["customers", "largest_share"]]).join(curated.set_index("merchant_abn"))
profiles["customers"] = profiles.customers.fillna(0)
features = {
    "Expected monthly return": ("expected_monthly_fees", "A${:,.0f}"),
    f"Customers over {week_count} weeks": ("customers", "{:,.0f}"),
    "Weekly return variability": ("weekly_cv", "{:.1%}"),
    "Largest consumer’s share": ("largest_share", "{:.2%}"),
    "Merchant take rate": ("take_rate", "{:.2f}%"),
    "Average transaction value": ("avg_value_per_transaction", "A${:,.0f}"),
    "Average consumer fraud score": ("avg_consumer_fraud_prob", "{:.2f}%"),
}
group_order = ["Always selected", "Sometimes selected", "Never selected"]
group_counts = profiles["Selection group"].value_counts().reindex(group_order, fill_value=0)
summary_medians = pd.DataFrame({g: {label: profiles.loc[profiles["Selection group"].eq(g), col].median() for label, (col, fmt) in features.items()} for g in group_order})
# Presentation uses conventional half-up rounding for whole-customer medians.
from decimal import Decimal, ROUND_HALF_UP
def format_median(label, value, fmt):
    if pd.isna(value):return "—"
    if label.startswith("Customers over"):
        return f"{Decimal(str(value)).quantize(Decimal('1'), rounding=ROUND_HALF_UP):,}"
    return fmt.format(value)
summary_display = pd.DataFrame({f"{g} ({group_counts[g]:,})": {label: format_median(label, summary_medians.loc[label, g], fmt) for label, (col, fmt) in features.items()} for g in group_order})
display(summary_display.rename_axis("Summary statistics (median)"))
missing_counts = profiles.groupby("Selection group")[[col for col, fmt in features.values()]].apply(lambda f: f.isna().sum()).reindex(group_order, fill_value=0)
if missing_counts.to_numpy().any():
    display(Markdown("**Missing attributes excluded from each median:**"))
    display(missing_counts)
assert group_counts.sum() == len(eligible)
assert group_counts["Always selected"] + group_counts["Sometimes selected"] == selected_union

display(Markdown("**Sometimes-selected merchants and their exact alpha memberships**"))
sometimes = profiles.loc[profiles["Selection group"].eq("Sometimes selected"), ["name", "selection_count"]].copy()
sometimes["Selected at α"] = alpha_memberships.reindex(sometimes.index)
display(sometimes.sort_values(["selection_count", "name"], ascending=[False, True]))

,Always selected (90),Sometimes selected (23),"Never selected (3,784)"
Summary statistics (median),,,
Expected monthly return,"A$14,185","A$7,136",A$237
Customers over 26 weeks,"6,956","1,469",136
Weekly return variability,9.5%,16.8%,53.8%
Largest consumer’s share,0.10%,0.33%,2.68%
Merchant take rate,5.54%,5.83%,4.47%
Average transaction value,A$279,A$677,A$302
Average consumer fraud score,4.83%,5.80%,4.88%


**Sometimes-selected merchants and their exact alpha memberships**

,name,selection_count,Selected at α
merchant_abn,,,
46012371285,A Ultricies Inc.,4,"100%, 99.5%, 99%, 98%"
61080648284,Lobortis Ultrices LLC,4,"99.5%, 99%, 98%, 95%"
49549583265,Luctus Et Incorporated,4,"100%, 99.5%, 99%, 98%"
82368304209,Nec Incorporated,4,"100%, 99.5%, 99%, 98%"
85529538986,Sed Neque Associates,4,"100%, 99.5%, 99%, 98%"
70009327857,Torquent Per Inc.,4,"99.5%, 99%, 98%, 95%"
38212167834,Magna Suspendisse Industries,3,"100%, 99.5%, 99%"
98518649381,Nullam Scelerisque Neque Corp.,3,"100%, 99.5%, 99%"
39649557865,Arcu Morbi Institute,2,"99.5%, 99%"


## 13. Changes from the original unadjusted-return selections

The table compares current adjusted selections with the preserved original exports. Return levels use different definitions; a decrease is a change of measurement, not a realised financial loss. Risk indices have different baselines, so compare old and new selections on the same adjusted basis using the supplied `old_and_new_on_adjusted_basis.csv`. Merchant substitutions are counts of incoming merchants (the same number leave for equal-sized portfolios).

In [11]:
original_dashboards = {label: pd.read_csv(PROJECT_ROOT / "handoff/final" / filename, dtype={"merchant_abn": "string"}) for label, filename in DASHBOARD_FILES.items()}
change_rows = []
for scope, current in dashboards.items():
    original = original_dashboards[scope]
    for alpha, rows in current.groupby("retention"):
        old = original.loc[np.isclose(original.retention, alpha)]
        if old.empty:continue
        incoming = set(rows.merchant_abn)-set(old.merchant_abn)
        outgoing = set(old.merchant_abn)-set(rows.merchant_abn)
        change_rows.append({"Scope":scope,"α (%)":alpha*100,"Incoming":len(incoming),"Outgoing":len(outgoing),"Original monthly return before fraud adjustment (AUD)":old.portfolio_expected_monthly_gross_fees.iloc[0],"Adjusted expected monthly return (AUD)":rows.portfolio_expected_monthly_adjusted_fees.iloc[0],"Adjusted risk index":rows.portfolio_risk_score.iloc[0],"Certified":rows.portfolio_certified.iloc[0]})
selection_changes = pd.DataFrame(change_rows)
display(selection_changes)
same_basis_path = PROJECT_ROOT / "handoff/fraud_adjusted/old_and_new_on_adjusted_basis.csv"
if same_basis_path.exists():
    display(Markdown("**Old and new selections evaluated on the same adjusted basis**"))
    display(pd.read_csv(same_basis_path))

,Scope,α (%),Incoming,Outgoing,Original monthly return before fraud adjustment (AUD),Adjusted expected monthly return (AUD),Adjusted risk index,Certified
0,Overall,95.000000,5,5,"1,583,307.021488","1,474,508.537992",96.168103,False
1,Overall,98.000000,3,3,"1,633,625.865226","1,520,697.987010",97.817023,True
2,Overall,99.000000,2,2,"1,649,905.124237","1,536,392.767695",98.601015,True
3,Overall,99.500000,2,2,"1,658,375.259911","1,544,100.827643",99.060518,True
4,Overall,100.000000,1,1,"1,666,399.534895","1,551,710.015613",100.000000,True
5,"Arts, Crafts, Gifts & Entertainment",95.000000,5,5,"756,726.101010","701,319.121139",94.893347,False
6,"Arts, Crafts, Gifts & Entertainment",98.000000,5,5,"780,527.631339","723,449.839623",96.430336,False
7,"Arts, Crafts, Gifts & Entertainment",99.000000,7,7,"788,437.927417","730,796.878911",97.120981,False
8,"Arts, Crafts, Gifts & Entertainment",99.500000,7,7,"792,396.924633","734,499.886800",97.679896,True
9,"Arts, Crafts, Gifts & Entertainment",100.000000,6,6,"796,374.837534","738,176.936041",100.000000,True


**Old and new selections evaluated on the same adjusted basis**

,scope,retention,old_selection_adjusted_fees,new_selection_adjusted_fees,old_selection_adjusted_cv,new_selection_adjusted_cv,old_selection_adjusted_hhi,new_selection_adjusted_hhi,old_selection_adjusted_score,new_selection_adjusted_score,old_selection_meets_new_floor,certified
0,global,0.950000,"1,488,396.810186","1,474,508.537992",0.029992,0.029335,0.000045,0.000045,96.865499,96.168103,True,False
1,global,0.980000,"1,529,314.632698","1,520,697.987010",0.030628,0.030324,0.000045,0.000045,98.186154,97.817023,True,True
2,global,0.990000,"1,542,648.581861","1,536,392.767695",0.030973,0.030883,0.000045,0.000045,98.987380,98.601015,True,True
3,global,0.995000,"1,544,828.836060","1,544,100.827643",0.030898,0.031024,0.000046,0.000045,99.168943,99.060518,True,True
4,global,1.000000,"1,551,676.143691","1,551,710.015613",0.031264,0.031376,0.000046,0.000046,99.837552,100.000000,False,True
5,tech,0.950000,"548,318.086827","545,673.772812",0.033988,0.033651,0.000046,0.000046,94.305862,93.775367,True,False
6,tech,0.980000,"563,976.978791","562,924.052497",0.034903,0.034812,0.000048,0.000048,96.913866,96.806865,True,False
7,tech,0.990000,"569,809.454824","568,669.172980",0.035601,0.035395,0.000048,0.000048,97.776490,97.595195,True,True
8,tech,0.995000,"572,155.610369","571,521.433821",0.035765,0.035738,0.000048,0.000048,98.470266,98.182007,True,True
9,tech,1.000000,"574,311.364005","574,383.114523",0.036656,0.036637,0.000049,0.000048,100.284039,100.000000,False,True
